## Generate responses for initial analysis

model: Qwen2.5-7B-Instruct

The output will serve as an input to initial analysis for decision methodology on what tokens get AV explanations.

In [ ]:
import os
os.environ["HF_HOME"] = "/workspace/nla_infer_010726/hf"
# fill in the token - DO NOT COMMIT!
os.environ["HF_TOKEN"] = "hf_xxxxxxxx"

In [ ]:
import json, yaml, torch
import pyarrow as pa, pyarrow.parquet as pq
from transformers import AutoModelForCausalLM, AutoTokenizer
from .autonotebook import tqdm as notebook_tqdm

In [ ]:
cfg = yaml.safe_load(open("prompts.yaml"))
C = cfg["config"]
LAYER = C["layer"]

tok = AutoTokenizer.from_pretrained(C["model"])
m = AutoModelForCausalLM.from_pretrained(C["model"], dtype=torch.bfloat16,
                                         device_map="cuda")

rows = []

In [ ]:
for p in cfg["prompts"]:
    enc = tok.apply_chat_template(
        [{"role": "user", "content": p["text"]}],
        add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to("cuda")
    
    prompt_len = enc["input_ids"].shape[1]

    for g in range(C["n_generations"]):
        # generate response
        out = m.generate(**enc, max_new_tokens=C["max_new_tokens"],
                         do_sample=True, temperature=C["temperature"])
        full = out[0]

        # one clean forward pass over the full sequence
        with torch.no_grad():
            fw = m(full.unsqueeze(0), output_hidden_states=True)
        h = fw.hidden_states[LAYER][0]                     # [T, d]
        h_norm = h.float().norm(dim=-1).cpu().tolist()

        # logprob of each ACTUAL next token: logits[t] predicts token[t+1]
        logprobs = torch.log_softmax(fw.logits[0].float(), dim=-1)
        tok_lp = [float("nan")] + [
            logprobs[t - 1, full[t]].item() for t in range(1, len(full))
        ]

        rows.append({
            "prompt_id": p["id"], "condition": p["condition"],
            "pair_id": p.get("pair_id"), "gen_idx": g,
            "prompt_text": p["text"],
            "response_text": tok.decode(full[prompt_len:], skip_special_tokens=True),
            "prompt_len": prompt_len,
            "token_ids": full.cpu().tolist(),
            "token_strs": [tok.decode([t]) for t in full.tolist()],
            "token_logprob": tok_lp,
            "h_norm": h_norm,
            "model": C["model"], "layer": LAYER,
            "gen_config": json.dumps(C),
        })
        print(f"{p['id']} gen{g}: {rows[-1]['response_text'][:70]!r}")

pq.write_table(pa.Table.from_pylist(rows, schema=schema),
               "/workspace/nla_probe/responses.parquet", compression="zstd")
print(f"wrote {len(rows)} rows")